# Walmart 周销售额预测 —— 最优模型 B_cluster (DTW + 层次聚类 + XGBoost)

本 Notebook 复现本项目在**无泄漏滚动 CV** 下的最优模型：

1. **DTW 聚类**：对门店 × 周 `log1p(销售额)` 序列计算 DTW 距离矩阵，用层次聚类并按**轮廓系数**自动定簇（K=5，约束每簇 ≥5 家店）；
2. **簇内建模**：每个簇训练一个浅层 XGBoost（`max_depth=3`），目标 `log1p`，Duan smearing 还原；
3. **评估**：滚动原点 CV + 末段留出，指标为 Walmart 官方 **WMAE（节假日×5）** 及 WAPE/RMSE/R²。

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

SEED = 42
np.random.seed(SEED)

CFG = {
    "target": "Weekly_Sales",
    "store_col": "Store",
    "date_col": "Date",
    "date_format": "%d-%m-%Y",
    "holiday_col": "Holiday_Flag",
    "exogenous": ["Temperature", "Fuel_Price", "CPI", "Unemployment", "Holiday_Flag"],
    "time_features": ["month", "week", "quarter"],
    "lag_features": [1, 2, 4, 52],
    "rolling_windows": [4, 8, 13],
    "n_splits": 5,
    "test_size": 13,
    "min_train": 52,
    "xgboost": dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=5,
                    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                    random_state=SEED, n_jobs=1, verbosity=0),
}
print("seed =", SEED, "| config ready")

In [ ]:
# 环境自检（Kaggle 标准引导）
import os, sys
print("Python:", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__)
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
def _has_columns(path):
    try:
        cols = set(pd.read_csv(path, nrows=0).columns)
        return {"Weekly_Sales", "Store", "Date"}.issubset(cols)
    except Exception:
        return False


def find_dataset():
    """依次尝试：Kaggle 挂载目录 -> kagglehub 下载 -> 本地回退。"""
    for path in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        if _has_columns(path):
            return path
    try:
        import kagglehub
        root = kagglehub.dataset_download("mikhail1681/walmart-sales")
        for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
            if _has_columns(path):
                return path
    except Exception as err:
        print("kagglehub 回退不可用:", err)
    for path in ["practice_1.csv", "../practice_1.csv", "/content/Walmart_Sales.csv",
                 "/kaggle/working/practice_1.csv"]:
        if os.path.exists(path) and _has_columns(path):
            return path
    raise FileNotFoundError("未找到包含 Weekly_Sales/Store/Date 的 CSV")


DATA_PATH = find_dataset()
print("使用数据:", DATA_PATH)

In [ ]:
def add_time_features(df):
    d = df[CFG["date_col"]]
    df = df.copy()
    df["year"] = d.dt.year
    df["month"] = d.dt.month
    df["quarter"] = d.dt.quarter
    df["week"] = d.dt.isocalendar().week.astype(int)
    return df


def add_history_features(df):
    """门店内滞后/滚动特征，仅用过去值，避免未来泄漏。"""
    df = df.sort_values([CFG["store_col"], CFG["date_col"]]).reset_index(drop=True).copy()
    y = np.log1p(df[CFG["target"]].to_numpy(float))
    names = [f"lag{l}" for l in CFG["lag_features"]] + \
            [f"roll_{k}{w}" for w in CFG["rolling_windows"] for k in ("mean", "std")]
    for n in names:
        df[n] = np.nan
    for _, idx in df.groupby(CFG["store_col"]).indices.items():
        idx = np.sort(idx)
        s = pd.Series(y[idx])
        for lag in CFG["lag_features"]:
            v = np.full(len(idx), np.nan)
            if lag < len(idx):
                v[lag:] = y[idx[:-lag]]
            df.loc[idx, f"lag{lag}"] = v
        for w in CFG["rolling_windows"]:
            sh = s.shift(1)
            df.loc[idx, f"roll_mean{w}"] = sh.rolling(w).mean().to_numpy()
            df.loc[idx, f"roll_std{w}"] = sh.rolling(w).std().to_numpy()
    return df, names


raw = pd.read_csv(DATA_PATH)
raw[CFG["date_col"]] = pd.to_datetime(raw[CFG["date_col"]], format=CFG["date_format"])
df = add_time_features(raw)
df, HISTORY_COLS = add_history_features(df)
print("数据规模:", df.shape, "| 门店", df[CFG["store_col"]].nunique(),
      "| 周", df[CFG["date_col"]].nunique())
print("历史特征:", HISTORY_COLS)

In [ ]:
def build_design(frame, include_store):
    cols = CFG["exogenous"] + CFG["time_features"] + HISTORY_COLS
    X = frame[cols].astype(float)
    if include_store:
        X = pd.concat([X, pd.get_dummies(frame[CFG["store_col"]], prefix="Store").astype(float)], axis=1)
    return X.reset_index(drop=True)


def dtw_distance(a, b):
    n, m = len(a), len(b)
    prev = np.full(m + 1, np.inf); prev[0] = 0.0
    for i in range(1, n + 1):
        cur = np.full(m + 1, np.inf)
        for j in range(1, m + 1):
            cur[j] = abs(a[i - 1] - b[j - 1]) + min(prev[j], cur[j - 1], prev[j - 1])
        prev = cur
    return prev[m]


def dtw_matrix(series):
    n = series.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = dtw_distance(series[i], series[j])
    return D


pivot = df.pivot(index=CFG["store_col"], columns=CFG["date_col"], values=CFG["target"]).sort_index()
stores = pivot.index.to_numpy()
series = np.log1p(pivot.to_numpy(float))          # 保留规模（log 水平）
D = dtw_matrix(series)
print("DTW 距离矩阵:", D.shape)


def select_k(D, k_min=2, k_max=8, min_size=5):
    best, best_sil, scores = None, -1.0, {}
    for k in range(k_min, k_max + 1):
        lab = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average").fit_predict(D)
        if np.bincount(lab).min() < min_size:      # 排除离群店单独成簇的退化解
            continue
        sil = silhouette_score(D, lab, metric="precomputed")
        scores[k] = sil
        if sil > best_sil:
            best, best_sil = k, sil
    return best, scores


K, SIL = select_k(D)
labels = AgglomerativeClustering(n_clusters=K, metric="precomputed", linkage="average").fit_predict(D)
store_to_cluster = dict(zip(stores.tolist(), labels.tolist()))
df["cluster"] = df[CFG["store_col"]].map(store_to_cluster)
print("自动定簇 K =", K, "| 轮廓系数:", {k: round(v, 3) for k, v in SIL.items()})
print("簇规模:", np.bincount(labels).tolist())

In [ ]:
def rolling_splits(n_dates, n_splits, test_size, min_train):
    out = []
    for i in range(n_splits):
        val_end = n_dates - (n_splits - 1 - i) * test_size
        val_start = val_end - test_size
        if val_start < min_train:
            continue
        out.append((np.arange(0, val_start), np.arange(val_start, val_end)))
    return out


def metrics(y, p, w):
    e = np.abs(y - p)
    return dict(mae=float(e.mean()),
                rmse=float(np.sqrt(np.mean((y - p) ** 2))),
                wmae=float(np.sum(w * e) / np.sum(w)),
                wape=float(e.sum() / np.abs(y).sum()),
                r2=float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)))


X_no = build_design(df, include_store=False)
y_true = df[CFG["target"]].to_numpy(float)
y_log = np.log1p(y_true)
w_all = np.where(df[CFG["holiday_col"]].to_numpy() == 1, 5.0, 1.0)
dates = np.sort(df[CFG["date_col"]].unique())
clusters = df["cluster"].to_numpy()


def grouped_predict(X, tr, va, group):
    """每个簇训练一个 XGBoost，Duan smearing 还原到原尺度。"""
    Xf = X.fillna(X.iloc[tr].median())
    pred = np.full(len(va), np.nan)
    trg, vag = group[tr], group[va]
    for g in np.unique(vag):
        a = tr[trg == g]
        b = np.flatnonzero(vag == g)
        model = XGBRegressor(**CFG["xgboost"])
        model.fit(Xf.iloc[a], y_log[a])
        factor = float(np.mean(np.exp(y_log[a] - model.predict(Xf.iloc[a]))))
        pred[b] = np.expm1(model.predict(Xf.iloc[va[b]]) + np.log(factor))
    return pred


splits = rolling_splits(len(dates), CFG["n_splits"], CFG["test_size"], CFG["min_train"])
rows, pt, pp, pw = [], [], [], []
for fold, (tp, vp) in enumerate(splits):
    tr = np.flatnonzero(df[CFG["date_col"]].isin(set(dates[tp])).to_numpy())
    va = np.flatnonzero(df[CFG["date_col"]].isin(set(dates[vp])).to_numpy())
    pred = grouped_predict(X_no, tr, va, clusters)
    m = metrics(y_true[va], pred, w_all[va])
    rows.append({"fold": fold, **m})
    pt.append(y_true[va]); pp.append(pred); pw.append(w_all[va])
    print(f"fold {fold}: WMAE={m['wmae']:,.0f}  WAPE={m['wape']:.4f}  R2={m['r2']:.4f}")

cv = pd.DataFrame(rows)
pooled = metrics(np.concatenate(pt), np.concatenate(pp), np.concatenate(pw))
print("\n滚动 CV 逐折均值: WMAE=%.0f  WAPE=%.4f  R2=%.4f"
      % (cv.wmae.mean(), cv.wape.mean(), cv.r2.mean()))
print("滚动 CV 汇总(合并): WMAE=%.0f  WAPE=%.4f  R2=%.4f"
      % (pooled["wmae"], pooled["wape"], pooled["r2"]))

In [ ]:
# 末段留出最后 13 周做无泄漏终验
split_date = dates[-CFG["test_size"]]
tr = np.flatnonzero((df[CFG["date_col"]] < split_date).to_numpy())
te = np.flatnonzero((df[CFG["date_col"]] >= split_date).to_numpy())
hold_pred = grouped_predict(X_no, tr, te, clusters)
hold = metrics(y_true[te], hold_pred, w_all[te])
print("末段留出(最后 %d 周):" % CFG["test_size"])
for k, v in hold.items():
    print(f"  {k} = {v:,.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ks = sorted(SIL)
axes[0].plot(ks, [SIL[k] for k in ks], "o-")
axes[0].axvline(K, color="red", ls="--", label=f"K={K}")
axes[0].set_title("DTW clustering silhouette"); axes[0].set_xlabel("n_clusters")
axes[0].set_ylabel("silhouette"); axes[0].legend()

axes[1].scatter(y_true[te], hold_pred, s=8, alpha=0.3)
lim = [0, max(y_true[te].max(), np.nanmax(hold_pred)) * 1.05]
axes[1].plot(lim, lim, "r--")
axes[1].set_title("Predicted vs actual (holdout)"); axes[1].set_xlabel("actual"); axes[1].set_ylabel("predicted")

axes[2].hist(y_true[te] - hold_pred, bins=50, color="#d62728", alpha=0.8)
axes[2].set_title("Residual distribution (holdout)"); axes[2].set_xlabel("actual - predicted")
plt.tight_layout(); plt.show()

In [ ]:
# 全局 XGBoost 的特征重要性（用于解释）
model_full = XGBRegressor(**CFG["xgboost"])
X_all = build_design(df, include_store=True).fillna(0)
model_full.fit(X_all, y_log)
imp = pd.Series(model_full.feature_importances_, index=X_all.columns).sort_values(ascending=False).head(15)[::-1]
plt.figure(figsize=(8, 6))
plt.barh(imp.index, imp.values, color="#1f77b4")
plt.title("XGBoost feature importance (gain)"); plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).head(10))

## 结论

- **B_cluster（DTW + 层次聚类 + XGBoost）** 在本数据上表现最优，且最差门店也最稳；
- 销售额首要驱动是**自身历史动量（lag1）与门店规模/季节性**，宏观变量贡献较小；
- 单店独立建模（143 样本/店）易过拟合，簇级是“样本量 vs 门店差异”的最优折中；

In [ ]:
sub = pd.DataFrame({
    "Id": df[CFG["store_col"]].to_numpy()[te].astype(str) + "_" +
          pd.to_datetime(df[CFG["date_col"]].to_numpy()[te]).strftime("%Y-%m-%d"),
    "Weekly_Sales": np.clip(hold_pred, 0, None),
}).sort_values("Id")
sub.to_csv("submission.csv", index=False)
print("已生成 submission.csv，行数:", len(sub))
sub.head()